# Seleção de features (Fase 3)

**Objetivo:** decidir quais colunas da camada Gold `features_aluno` entram no modelo
supervisionado, como cada uma deve ser tratada no pré-processamento, e registrar
o motivo de cada decisão.

**Entrada:** `data/sample/features_aluno` (1 linha por aluno avaliado em 2024).
**Saída:** a lista final de features, usada pelo notebook `03_modelagem_supervisionada`.


## 1. Bibliotecas e configuracões

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", 50)

In [2]:
CAMINHO_GOLD = "../data/sample/features_aluno"

df = pd.read_parquet(CAMINHO_GOLD)
df.shape

(1610754, 35)

## 2. Raio-x das colunas

Tipo, número de valores distintos e percentual de nulos de cada coluna.
É a base para separar alvo, identificadores, constantes e features.

In [3]:
raio_x = pd.DataFrame({
    "tipo": df.dtypes,
    "valores_distintos": df.nunique(),
    "pct_nulos": (df.isna().mean() * 100).round(3),
}).sort_values("valores_distintos")

raio_x

,tipo,valores_distintos,pct_nulos
ano,int32,1,0.000
rede_codigo,str,1,0.000
uf_anomala,bool,2,0.000
alfabetizado,bool,2,0.000
mun_elegivel_meta,object,2,0.001
regiao,str,5,0.000
caderno,str,22,0.000
sigla_uf,str,25,0.000
mun_idade_mediana,float64,35,0.000
mun_total_escolas,int32,134,0.000


In [4]:
## Saber qual o valor está nas colunas de ano e rede_codig:

df[["ano", "rede_codigo"]].drop_duplicates()


,ano,rede_codigo
0,2024,3


In [5]:
#avaliando a variável "caderno"
df.groupby("caderno")["alfabetizado"].agg(["mean", "count"]).sort_values("count")

,mean,count
caderno,,
43,0.454545,11
14,0.598534,74081
19,0.592374,74274
13,0.600269,74330
9,0.591909,74334
21,0.595269,74368
17,0.589278,74369
11,0.594284,74382
18,0.583775,74402


In [6]:
# avaliando a coluna"uf_anomala"
pd.crosstab(df["uf_anomala"], df["sigla_uf"] == "RS")

sigla_uf,False,True
uf_anomala,,
False,1549701,0
True,0,61053


**Conclusões do raio-x:**

- **Variáveis Constantes:** `ano` (sempre 2024) e `rede_codigo` (sempre "3") têm um único valor
  em todas as linhas, portanto não carregam informação para o modelo e serão excluídas.

- **Variáveis Binárias:** `alfabetizado` é uma variável binária e é justamente o **alvo** do modelo. `uf_anomala` e `mun_elegivel_meta` são candidatas a feature. Um ponto importante é sobr a variável `mun_elegivel_meta` que tem tipo `object` por causa de 0,001% de nulos e precisará de imputação.

- **Variáveis Categóricas de baixa cardinalidade:** `regiao` (5), `sigla_uf` (25) e `caderno` (22)
  são texto e precisarão de encoding.
 
- **Identificadores:** `id_aluno` tem um valor por linha (chave única); `id_escola` (36.920)
  e `id_municipio` (5.450) identificam grupos. Nenhum entra como feature. 
  O `id_municipio` será usado para separar treino e teste.

- **Colunas municipais:** `mun_pib_per_capita` tem exatamente 5.450 valores, um por município.

- **Nulos:** `mun_distancia_meta_anterior` e `mun_meta_ano_alvo` (2,6%, mesmo percentual,
  provavelmente as mesmas linhas), `mun_taxa_ano_anterior` (0,5%), as quatro
  `mun_pct_va_*` (0,006%) e `mun_elegivel_meta` (0,001%). 
  Com todos abaixo de 3% as colunas serão mantidas e os vazios serão tratados por imputação.


## 3. O alvo: `alfabetizado`

**Definição:** o aluno é considerado alfabetizado quando sua proficiência no Saeb é
maior ou igual a 743 pontos (ponto de corte do Indicador Criança Alfabetizada).
Fonte: README do projeto, seção 3 (a regra foi verificada na Silver da Fase 2).

In [7]:
contagem = df["alfabetizado"].value_counts()
percentual = df["alfabetizado"].value_counts(normalize=True).mul(100).round(1)

pd.DataFrame({"alunos": contagem, "percentual": percentual})

,alunos,percentual
alfabetizado,,
True,956343,59.4
False,654411,40.6


**Decisão: inverter o alvo**

No scikit-learn, precisão, recall e F1 são calculados em relação à **classe positiva**,
que por padrão é a classe 1. Na base, o 1 (`True`) é o aluno **alfabetizado**. Treinado
assim, o recall responderia "de todas as crianças alfabetizadas, quantas o modelo
encontrou?", e não é essa a pergunta da secretaria de educação.

O objetivo é **encontrar as crianças em risco**. O erro mais grave é o falso negativo:
uma criança não alfabetizada que o modelo classifica como alfabetizada e que, por isso,
fica fora da ação da política pública.

Por isso o alvo é invertido: **1 = não alfabetizado (risco)** e **0 = alfabetizado**.
A acurácia não é afetada pela inversão; precisão, recall e F1 passam a medir a
capacidade do modelo de encontrar a classe de risco.

In [8]:
# alvo invertido: 1 = não alfabetizado (classe de risco), 0 = alfabetizado
y = (~df["alfabetizado"]).astype(int).rename("nao_alfabetizado")

# prova da inversão: cruza o alvo original com o novo
pd.crosstab(df["alfabetizado"], y, margins=True)

nao_alfabetizado,0,1,All
alfabetizado,,,
False,0,654411,654411
True,956343,0,956343
All,956343,654411,1610754


In [9]:
rotulos = {0: "alfabetizado", 1: "não alfabetizado (risco)"}
y.value_counts().rename(index=rotulos)

nao_alfabetizado
alfabetizado                956343
não alfabetizado (risco)    654411
Name: count, dtype: int64

**Conclusões sobre o alvo:**

- **Balanceamento:** 59,4% dos alunos são alfabetizados e 40,6% não alfabetizados.
  A base é relativamente balanceada, portanto não são necessárias técnicas de balanceamento
  artificial, mas a divisão entre treino e teste deve preservar essa proporção.
- **Baseline ingênuo:** um modelo que responde sempre "alfabetizado" acerta 59,4%
  (acurácia), mas não identifica nenhuma criança em risco (recall de 0% na classe 1).
  O modelo final precisa superar esse patamar, e a acurácia sozinha não basta para
  avaliá-lo.
- **Métrica prioritária:** o erro mais grave é o falso negativo, a criança em risco
  que o modelo classifica como alfabetizada e que fica fora da política pública.
  Por isso a métrica prioritária é o **recall da classe de risco**, acompanhado da
  **precisão** e do **F1**: um modelo que apontasse risco para todos teria recall de
  100% e nenhuma utilidade prática.

## 4. Redundância entre as features

Verifica se alguma coluna numérica municipal repete a informação de outra
(multicolinearidade). A redundância prejudica pouco a previsão, mas distorce a
interpretação (Feature Importance e SHAP), porque o modelo divide a importância
entre colunas que dizem o mesmo.

As correlações são calculadas com **uma linha por município**: as colunas `mun_` se
repetem em todos os alunos do município, e usar os 1,6 milhão de alunos daria a
municípios grandes um peso desproporcional.

In [10]:
municipios = df.drop_duplicates(subset="id_municipio")
numericas_mun = [c for c in df.columns if c.startswith("mun_") and c != "mun_elegivel_meta"]

print(f"Municípios: {len(municipios)} | colunas numéricas municipais: {len(numericas_mun)}")

Municípios: 5450 | colunas numéricas municipais: 23


In [11]:
# teste da conta exata: distância = taxa do ano anterior - meta
diferenca = (
    municipios["mun_taxa_ano_anterior"]
    - municipios["mun_meta_ano_alvo"]
    - municipios["mun_distancia_meta_anterior"]
)
diferenca.abs().max()

np.float64(0.0)

In [12]:
# pares de colunas mais correlacionados (valor absoluto)
corr = municipios[numericas_mun].corr().abs()

pares = (
    corr.where(np.triu(np.ones(corr.shape, dtype=bool), k=1))
        .stack()
        .sort_values(ascending=False)
)
pares.head(10)

mun_taxa_ano_anterior          mun_meta_ano_alvo                   0.978675
mun_idade_mediana              mun_indice_envelhecimento           0.927986
mun_taxa_ano_anterior          mun_distancia_meta_anterior         0.847766
mun_moradores_por_domicilio    mun_idade_mediana                   0.806024
mun_alunos_por_docente         mun_alunos_por_turma                0.733688
mun_taxa_alfabetizacao_adulta  mun_pct_va_administracao_publica    0.732840
mun_meta_ano_alvo              mun_distancia_meta_anterior         0.720743
mun_total_escolas              mun_populacao                       0.691818
mun_pct_rural                  mun_pct_transporte                  0.652632
mun_moradores_por_domicilio    mun_indice_envelhecimento           0.642833
dtype: float64

**Critério (definido antes da análise):**

| Relação | Decisão |
|---|---|
| Conta exata ou correlação acima de 0,9 | Manter só uma das colunas |
| Correlação entre 0,7 e 0,9 | Manter as duas e considerar a relação na interpretação |
| Correlação abaixo de 0,7 | Nada a fazer |

**Resultados:**

- `mun_meta_ano_alvo` tem correlação de 0,98 com `mun_taxa_ano_anterior` e pode ser
  calculada exatamente a partir das outras duas colunas de meta (diferença máxima 0).
  **Será removida**; ficam a taxa do ano anterior (onde o município estava) e a
  distância da meta (quanto faltava, medida de pressão institucional segundo o README).
- `mun_idade_mediana` tem correlação de 0,93 com `mun_indice_envelhecimento` e de 0,81
  com `mun_moradores_por_domicilio`. **Será removida**: resolve as duas relações, e tem
  só 35 valores distintos, contra 4.105 do índice de envelhecimento.
- Relações moderadas mantidas (cuidado na interpretação do SHAP):
  `mun_alunos_por_docente` × `mun_alunos_por_turma` (0,73) e
  `mun_taxa_alfabetizacao_adulta` × `mun_pct_va_administracao_publica` (0,73).

**Observação metodológica:** a correlação compara duas colunas por vez e não detecta
uma coluna que é combinação de várias. `mun_distancia_meta_anterior` tem correlação de
0,85 com a taxa e 0,72 com a meta, nenhuma próxima de 1, mas é exatamente a diferença
entre as duas. Por isso foi necessário o teste da conta exata.

## 5. Colunas excluídas do modelo

| Coluna | Grupo | Motivo | Evidência |
|---|---|---|---|
| `alfabetizado` | Alvo | É a variável que o modelo prevê. Sai das features e vira o `y`, invertido (1 = risco). | Seção 3 |
| `id_aluno` | Identificador | Identifica cada aluno (um valor por linha). Não carrega informação que se generalize para alunos novos. | Seção 2 (raio-x) |
| `id_escola` | Identificador | Identifica a escola. É um rótulo sem significado próprio, com 36.920 valores. | Seção 2 (raio-x) |
| `id_municipio` | Identificador | Identifica o município. Não entra como feature, mas é usado para dividir treino e teste por município (`COLUNA_GRUPO`). | Seção 2 (raio-x) |
| `ano` | Constante | Tem o mesmo valor em todas as linhas. Sem variação, não carrega informação. | Seção 2 (raio-x) |
| `rede_codigo` | Constante | Tem o mesmo valor em todas as linhas. Sem variação, não carrega informação. | Seção 2 (raio-x) |
| `peso_aluno` | Peso amostral | Indica quantos alunos da população cada avaliado representa. Descreve o desenho da avaliação, não a criança nem o seu contexto. Pode ser usado como peso no treino (`sample_weight`), decisão do notebook 03. | README, seção 3; correlação no `01_eda`, seção 7 |
| `caderno` | Descartada na seleção | Não está disponível no momento da previsão: o caderno só é sorteado no dia da prova, e o modelo deve sinalizar o risco antes dela. Além disso, os cadernos são desenhados para ter dificuldade equivalente, e os dados confirmam: a taxa de alfabetização quase não varia entre cadernos com cerca de 70 mil alunos cada (o caderno 43, com 11 alunos, é exceção sem relevância estatística). | Seção 2 (raio-x)|
| `uf_anomala` | Descartada na seleção | Redundante com `sigla_uf`: marca exatamente os 61.053 alunos do RS, e nenhum fora dele. Após o encoding, `sigla_uf_RS` carrega a mesma informação. A exclusão não remove o RS da base. | Seção 2 (cruzamento com `sigla_uf`) |
| `mun_meta_ano_alvo` | Redundante | Correlação de 0,98 com `mun_taxa_ano_anterior`; é calculável a partir das outras duas colunas de meta. | Seção 4 |
| `mun_idade_mediana` | Redundante | Correlação de 0,93 com `mun_indice_envelhecimento` e 0,81 com `mun_moradores_por_domicilio`; menos detalhada (35 valores distintos). | Seção 4 |


In [13]:
ALVO = "alfabetizado"
COLUNA_GRUPO = "id_municipio"   # usada para separar treino e teste por município

IDENTIFICADORES = ["id_aluno", "id_escola", "id_municipio"]
CONSTANTES = ["ano", "rede_codigo"]
PESO_AMOSTRAL = ["peso_aluno"]
DESCARTADAS_NA_SELECAO = ["caderno", "uf_anomala"]          # evidências na seção 2
REDUNDANTES = ["mun_meta_ano_alvo", "mun_idade_mediana"]   # evidências na seção 4

excluidas = (
    [ALVO] + IDENTIFICADORES + CONSTANTES + PESO_AMOSTRAL
    + DESCARTADAS_NA_SELECAO + REDUNDANTES
)
candidatas = [c for c in df.columns if c not in excluidas]

print(f"Excluídas: {len(excluidas)}")
print(f"Candidatas a feature: {len(candidatas)}")

Excluídas: 11
Candidatas a feature: 24


## 6. Features do modelo

In [14]:
CATEGORICAS = ["regiao", "sigla_uf"]
BINARIAS = ["mun_elegivel_meta"]

NUMERICAS = [c for c in candidatas if c not in CATEGORICAS + BINARIAS]
FEATURES = NUMERICAS + CATEGORICAS + BINARIAS

print(f"Numéricas:   {len(NUMERICAS)}")
print(f"Categóricas: {len(CATEGORICAS)}")
print(f"Binárias:    {len(BINARIAS)}")
print(f"Total de features: {len(FEATURES)}")

# conferência: as features são exatamente as candidatas, sem faltar nem repetir
assert sorted(FEATURES) == sorted(candidatas)


Numéricas:   21
Categóricas: 2
Binárias:    1
Total de features: 24


**Ponto em aberto (a discutir com o grupo):** a EDA trata a taxa mais baixa do RS
(45,3% contra 59,9% no restante do país) como possível erro na origem do dado. Uma
hipótese alternativa são as enchentes de maio de 2024 no estado, que fecharam escolas
por semanas. Se o efeito for real, excluir o RS removeria justamente os alunos com
maior risco. Por isso o RS foi mantido na base.
